# BÀI TẬP IPYWIDGETS


## Đề Bài


Luyện tập: áp dụng interactive widgets cho phân tích EDA dữ liệu chuyến bay.


## Script


### Using


In [ ]:
from pathlib import Path

import pandas as pd
import ipywidgets as widgets
from IPython.display import display
import matplotlib.pyplot as plt

### Fields


In [ ]:
# Khai báo tập trung tên cột và đường dẫn để các biểu đồ dùng cùng một nguồn dữ liệu đã chuẩn hóa.
DATA_DIR = Path('data')

CN_MINUTE = 'minute'
CN_HOUR = 'hour'
CN_DAY = 'day'
CN_MONTH = 'month'
CN_YEAR = 'year'
CN_DEP_TIME = 'dep_time'
CN_DEP_DELAY = 'dep_delay'
CN_ARR_TIME = 'arr_time'
CN_ARR_DELAY = 'arr_delay'
CN_CARRIER = 'carrier'
CN_TAILNUM = 'tailnum'
CN_FLIGHT = 'flight'
CN_ORIGIN = 'origin'
CN_DEST = 'dest'
CN_AIR_TIME = 'air_time'
CN_DISTANCE = 'distance'

**Tóm tắt:** Các hằng số giúp notebook tránh hard-code tên cột nhiều lần và đọc file `nycflights.csv` từ thư mục `data/`.


### Common


In [ ]:
# Chuẩn hóa chuỗi để các nhóm carrier/origin/dest không bị tách do khoảng trắng hoặc chữ thường/chữ hoa.
def normalize_text(series):
    return series.astype('string').str.strip().str.upper()


# Chuyển cột số sang kiểu Int64 nullable để giữ được giá trị thiếu nhưng vẫn so sánh/tính toán được.
def to_nullable_int(series):
    return pd.to_numeric(series, errors='coerce').astype('Int64')


# Định dạng tên cột thành nhãn dễ đọc trên biểu đồ.
def format_header(column_name):
    return column_name.replace('_', ' ').title()


# Đếm số bản ghi theo nhóm cho các biểu đồ số lượng chuyến bay.
def group_count(dataframe, group_column, value_column):
    return dataframe.groupby(group_column)[value_column].count()


# Tính trung bình delay khởi hành và delay đến theo một nhóm phân loại.
def group_delay_mean(dataframe, group_column):
    return dataframe.groupby(group_column)[[CN_DEP_DELAY, CN_ARR_DELAY]].mean().rename(
        columns={
            CN_DEP_DELAY: format_header(CN_DEP_DELAY),
            CN_ARR_DELAY: format_header(CN_ARR_DELAY),
        }
    )


# Điều kiện lọc chuỗi hợp lệ, dùng sau bước normalize_text.
def valid_text(series):
    return series.notna() & ~series.eq('')

**Tóm tắt:** Các hàm dùng chung gom phần làm sạch, đặt nhãn và groupby. Việc này giảm lặp code giữa các biểu đồ interactive widgets.


### Processing


In [ ]:
# Đọc dữ liệu chuyến bay từ thư mục data.
flights_df = pd.read_csv(DATA_DIR / 'nycflights.csv')

In [ ]:
# Tạo bản dữ liệu đã làm sạch để mọi phép lọc và groupby dùng cùng giá trị chuẩn hóa.
flights_clean_df = flights_df.copy()

text_columns = [CN_CARRIER, CN_TAILNUM, CN_ORIGIN, CN_DEST]
int_columns = [
    CN_MINUTE,
    CN_HOUR,
    CN_DAY,
    CN_MONTH,
    CN_YEAR,
    CN_DEP_TIME,
    CN_DEP_DELAY,
    CN_ARR_TIME,
    CN_ARR_DELAY,
    CN_FLIGHT,
    CN_AIR_TIME,
    CN_DISTANCE,
]

for column in text_columns:
    flights_clean_df[column] = normalize_text(flights_clean_df[column])

for column in int_columns:
    flights_clean_df[column] = to_nullable_int(flights_clean_df[column])

**Tóm tắt:** Notebook giữ `flights_df` làm dữ liệu thô và dùng `flights_clean_df` cho phân tích. Cách này tránh lỗi trước đây khi lọc bằng Series đã chuẩn hóa nhưng groupby lại dùng cột gốc chưa chuẩn hóa.


In [ ]:
# Request 1: so sánh số chuyến trễ/không trễ theo sân bay xuất phát.
origin_delay_df = flights_clean_df.loc[
    flights_clean_df[CN_DEP_DELAY].notna() & valid_text(flights_clean_df[CN_ORIGIN])
]

origin_flight_groups = {
    'Delayed': origin_delay_df.loc[origin_delay_df[CN_DEP_DELAY] > 0],
    'On-time': origin_delay_df.loc[origin_delay_df[CN_DEP_DELAY] <= 0],
}


def plot_flight_status_by_origin(flight_type):
    selected_df = origin_flight_groups[flight_type]
    flight_counts = group_count(selected_df, CN_ORIGIN, CN_DEP_DELAY)
    flight_counts.index.name = format_header(CN_ORIGIN)

    fig, ax = plt.subplots(figsize=(8, 5))
    flight_counts.plot.bar(rot=0, color='#4C78A8', ax=ax)
    ax.set_xlabel(format_header(CN_ORIGIN))
    ax.set_ylabel('Số lượng')
    ax.set_title(f'Số chuyến bay {"trễ" if flight_type == "Delayed" else "không trễ"} theo sân bay')
    plt.tight_layout()
    plt.show()


origin_status_widget = widgets.interactive(
    plot_flight_status_by_origin,
    flight_type=widgets.RadioButtons(
        options=['Delayed', 'On-time'],
        description='Flight type:',
        value='Delayed',
    ),
)
display(origin_status_widget)

**Tóm tắt:** Widget đầu tiên cho phép chuyển giữa chuyến bay trễ và không trễ để xem phân bố theo sân bay xuất phát.


In [ ]:
# Request 2: so sánh số chuyến trễ/không trễ theo hãng hàng không.
carrier_delay_df = flights_clean_df.loc[
    flights_clean_df[CN_DEP_DELAY].notna() & valid_text(flights_clean_df[CN_CARRIER])
]

carrier_flight_groups = {
    'Delayed': carrier_delay_df.loc[carrier_delay_df[CN_DEP_DELAY] > 0],
    'On-time': carrier_delay_df.loc[carrier_delay_df[CN_DEP_DELAY] <= 0],
}


def plot_flight_status_by_carrier(flight_type):
    selected_df = carrier_flight_groups[flight_type]
    flight_counts = group_count(selected_df, CN_CARRIER, CN_DEP_DELAY)
    flight_counts.index.name = format_header(CN_CARRIER)

    fig, ax = plt.subplots(figsize=(12, 5))
    flight_counts.plot.bar(rot=0, color='#59A14F', ax=ax)
    ax.set_xlabel(format_header(CN_CARRIER))
    ax.set_ylabel('Số lượng')
    ax.set_title(f'Số chuyến bay {"trễ" if flight_type == "Delayed" else "không trễ"} theo hãng hàng không')
    plt.tight_layout()
    plt.show()


carrier_status_widget = widgets.interactive(
    plot_flight_status_by_carrier,
    flight_type=widgets.RadioButtons(
        options=['Delayed', 'On-time'],
        description='Flight type:',
        value='Delayed',
    ),
)
display(carrier_status_widget)

**Tóm tắt:** Widget thứ hai dùng cùng logic với sân bay nhưng đổi trục nhóm sang `carrier`, giúp so sánh hãng nào có nhiều chuyến trễ hơn.


In [ ]:
# Request 3: chuẩn bị dữ liệu có đủ delay khởi hành, delay đến, origin và carrier.
delay_df = flights_clean_df.loc[
    flights_clean_df[CN_DEP_DELAY].notna()
    & flights_clean_df[CN_ARR_DELAY].notna()
    & valid_text(flights_clean_df[CN_ORIGIN])
    & valid_text(flights_clean_df[CN_CARRIER])
]

mean_delays_by_origin = group_delay_mean(delay_df, CN_ORIGIN)
mean_delays_by_origin.index.name = format_header(CN_ORIGIN)

mean_delays_by_carrier = group_delay_mean(delay_df, CN_CARRIER)
mean_delays_by_carrier.index.name = format_header(CN_CARRIER)

In [ ]:
# Request 3.1: xem delay trung bình theo sân bay.
def plot_average_delay_by_origin(delay_column):
    fig, ax = plt.subplots(figsize=(8, 5))
    mean_delays_by_origin[delay_column].plot(marker='o', color='#F28E2B', ax=ax)
    ax.set_xlabel(format_header(CN_ORIGIN))
    ax.set_ylabel('Delay (minutes)')
    ax.set_title(f'{delay_column} trung bình theo sân bay')
    plt.tight_layout()
    plt.show()


origin_delay_widget = widgets.interactive(
    plot_average_delay_by_origin,
    delay_column=widgets.RadioButtons(
        options=[format_header(CN_DEP_DELAY), format_header(CN_ARR_DELAY)],
        description='Delay type:',
        value=format_header(CN_DEP_DELAY),
    ),
)
display(origin_delay_widget)

**Tóm tắt:** Widget delay theo sân bay cho thấy độ trễ trung bình khi khởi hành và khi đến, giúp phát hiện sân bay có mức delay nổi bật.


In [ ]:
# Request 3.2: xem delay trung bình theo hãng hàng không.
def plot_average_delay_by_carrier(delay_column):
    fig, ax = plt.subplots(figsize=(12, 5))
    mean_delays_by_carrier[delay_column].plot(marker='o', color='#E15759', ax=ax)
    ax.set_xlabel(format_header(CN_CARRIER))
    ax.set_ylabel('Delay (minutes)')
    ax.set_title(f'{delay_column} trung bình theo hãng hàng không')
    plt.tight_layout()
    plt.show()


carrier_delay_widget = widgets.interactive(
    plot_average_delay_by_carrier,
    delay_column=widgets.RadioButtons(
        options=[format_header(CN_DEP_DELAY), format_header(CN_ARR_DELAY)],
        description='Delay type:',
        value=format_header(CN_DEP_DELAY),
    ),
)
display(carrier_delay_widget)

**Tóm tắt:** Widget delay theo hãng hàng không dùng cùng bộ dữ liệu delay đã làm sạch, nhưng đổi nhóm phân tích sang `carrier`.


In [ ]:
# Request 4: so sánh tổng, trung bình và trung vị quãng đường bay theo hãng.
distance_df = flights_clean_df.loc[
    flights_clean_df[CN_DISTANCE].notna()
    & (flights_clean_df[CN_DISTANCE] > 0)
    & valid_text(flights_clean_df[CN_CARRIER])
]

distance_summary_by_carrier = distance_df.groupby(CN_CARRIER)[CN_DISTANCE].agg(
    Sum='sum',
    Mean='mean',
    Median='median',
)
distance_summary_by_carrier.index.name = format_header(CN_CARRIER)


def plot_distance_by_carrier(metric):
    fig, ax = plt.subplots(figsize=(12, 5))
    distance_summary_by_carrier[metric].plot.bar(rot=0, color='#B07AA1', ax=ax)
    ax.set_xlabel(format_header(CN_CARRIER))
    ax.set_ylabel('Distance')
    ax.set_title(f'{metric} quãng đường bay theo hãng hàng không')
    if metric == 'Sum':
        ax.ticklabel_format(style='plain', axis='y')
    plt.tight_layout()
    plt.show()


distance_widget = widgets.interactive(
    plot_distance_by_carrier,
    metric=widgets.RadioButtons(
        options=['Sum', 'Mean', 'Median'],
        description='Select:',
        value='Sum',
    ),
)
display(distance_widget)

**Tóm tắt:** Widget cuối cùng tập trung vào quãng đường bay theo hãng và tránh dùng biến tên `type`, nhờ đó code không che khuất built-in của Python.


## Tổng kết

Notebook đã áp dụng interactive widgets cho các lát cắt EDA chính của dữ liệu chuyến bay: số chuyến trễ/không trễ theo sân bay và hãng, delay trung bình theo sân bay và hãng, cùng thống kê quãng đường bay theo hãng. Dữ liệu được đọc từ `data/`, làm sạch một lần trong `flights_clean_df`, sau đó tái sử dụng nhất quán cho toàn bộ biểu đồ.
